In [6]:
import os
import re
import numpy as np
import matplotlib.pyplot as plt
import netCDF4 as n
import glob
import xarray as xr
import seaborn as sns
import matplotlib.lines as mlines
import pandas as pd
import function_sampling_Thermal_forcing_files as fn
from matplotlib.colors import ListedColormap, BoundaryNorm
from scipy.stats import linregress
from collections import Counter
sns.set_context("paper")
sns.set_style("whitegrid")
computed_name = 'ComputedScalarsHelene'
variable2 = 'iareagr'

In [7]:
 
mnt_pth = '/Volumes/CrucialX8/computing/data/antarctica/' #mount path
pth_imip6hack= mnt_pth + 'ismip6_hackathon/ComputedScalars/' #write folder
pth_ismip6 =mnt_pth + 'ismip6_hackathon/' #read folder
pth_helene =mnt_pth + 'ismip6_hackathon/ComputedScalarsHelene/'
 
expnames = ['expAE02', 'expAE03', 'expAE04', 'expAE05']
removeFileID = ['IMAU_UFEMISM1', 'IMAU_UFEMISM2', 'IMAU_UFEMISM3', 'IMAU_UFEMISM4',
               'DOE_MALI_4km', 'DOE_MALI_8km_Ant95', 'DOE_MALI_8km_AntMean'] # specify models to remove
 
metadata_file = 'Metadata.txt'
# Generate loop_info DataFrame

In [8]:
def get_model(string,exp):
    return string.split('/')[-1].split('AIS_')[-1].split('_'+str(exp))[0]

In [9]:
dic_area_of_interest = {}
dic_area_of_interest['FilchnerRonne'] = ['_sector_5','_sector_11']
dic_area_of_interest['Ross'] = ['_sector_2','_sector_6']
dic_area_of_interest['Amundsen'] = ['_sector_4']


In [10]:
# avg and abs 2100
models=[]
paths =[]
exps =[]
for exp in expnames:
    lst = glob.glob(pth_helene+f'{exp}/shelfmelt/*.nc')
    lst.sort()

    for name in lst:
        model = get_model(name,exp)
        if model in removeFileID:
            continue
        else:
            models.append(model)
            paths.append(name)
            exps.append(exp)
df = pd.DataFrame({ 'Experiment': exps,'Model': models, 'Path': paths}) 
for i in range(1,19):
    df['sector_'+str(i)]=0
for i in range(1,4):
    df['region_'+str(i)]=0
for key in dic_area_of_interest.keys():
    df[key]=0
df['AIS']=0
df2 = df.copy()
for i in range (df.shape[0]):
    pth_way = df.iloc[i].Path
    #get shelf area
    pth_area = f'{pth_helene}/{df.iloc[i].Experiment}/iareagr/computed_iareagr_AIS_{df.iloc[i].Model}_{df.iloc[i].Experiment}.nc'
    pth_tf = f'{pth_imip6hack}/{df.iloc[i].Experiment}/thermalforcing/computed_thermalforcing_AIS_{df.iloc[i].Model}_{df.iloc[i].Experiment}.nc'


    #thermal foring 
#     d3 = xr.open_dataset(pth_tf,decode_times=False )

    # Melting
    d = xr.open_dataset(pth_way,decode_times=False )
    # Area
    d2 = xr.open_dataset(pth_area,decode_times=False )

    
    ind = np.where((d.time.values >=2070) & (d.time.values <= 2100))
    
    
    tmin = np.min([d.time.size,d2.time.size])
    #melting per area
    y =d2[variable2].values[:tmin] #GT/m^3 yr

   
    yais =y[ind].mean()
  
    
    df['AIS'].iloc[i]= (yais-  y[0])/y[0]*100
   

    for j in range(1,19):
        y =d2[f'{variable2}_sector_'+str(j)].values[:tmin]
        mnan = np.isnan(y)
        
        
       
        df['sector_'+str(j)].iloc[i] = (y[ind].mean()-  y[0])/  y[0]*100
        
    for j in range(1,4):
        y =d2[f'{variable2}_region_'+str(j)].values[:tmin]
        mnan = np.isnan(y)
        
        
       
        df['region_'+str(j)].iloc[i] = (y[ind].mean()-  y[0])/  y[0]*100
    for key in dic_area_of_interest.keys():
        sectors = dic_area_of_interest[key].copy()
        ys = np.zeros((len(sectors),tmin))
        for j,sector in enumerate(sectors):
            ys[j,:] = d2[f'{variable2}{sector}'].values[:tmin].copy()
        sumy = np.sum(ys.copy(),axis = 0)
        df[key].iloc[i]= (np.nanmean(sumy[ind]).copy()-  sumy[0])/  sumy[0]*100
        
        

        
        
#         if (j==1 and i ==4):
#             print(d2['iareafl_sector_'+str(j)].values[:tmin])
#             break
df.to_csv(f'tables/pchange_{variable2}_2100.csv', index=False)



/var/folders/ph/m61jd3r90t190p6qvbps3yqct2hr65/T/ipykernel_67707/243380356.py:53: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['AIS'].iloc[i]= (yais-  y[0])/y[0]*100
/var/folders/ph/m61jd3r90t190p6qvbps3yqct2hr65/T/ipykernel_67707/243380356.py:62: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['sector_'+str(j)].iloc[i] = (y[ind].mean()-  y[0])/  y[0]*100
/var/folders/ph/m61jd3r90t190p6qvbps3yqct2hr65/T/ipykernel_67707/243380356.py:70: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_g

In [15]:
# avg and abs 2100
models=[]
paths =[]
exps =[]
for exp in expnames:
    lst = glob.glob(pth_helene+f'{exp}/shelfmelt/*.nc')
    lst.sort()

    for name in lst:
        model = get_model(name,exp)
        if model in removeFileID:
            continue
        else:
            models.append(model)
            paths.append(name)
            exps.append(exp)
df = pd.DataFrame({ 'Experiment': exps,'Model': models, 'Path': paths}) 
for i in range(1,19):
    df['sector_'+str(i)]=0
for i in range(1,4):
    df['region_'+str(i)]=0
for key in dic_area_of_interest.keys():
    df[key]=0
df['AIS']=0
df2 = df.copy()
for i in range (df.shape[0]):
    pth_way = df.iloc[i].Path
    #get shelf area
    pth_area = f'{pth_helene}/{df.iloc[i].Experiment}/iareagr/computed_iareagr_AIS_{df.iloc[i].Model}_{df.iloc[i].Experiment}.nc'
    pth_tf = f'{pth_imip6hack}/{df.iloc[i].Experiment}/thermalforcing/computed_thermalforcing_AIS_{df.iloc[i].Model}_{df.iloc[i].Experiment}.nc'


    #thermal foring 
#     d3 = xr.open_dataset(pth_tf,decode_times=False )

    # Melting
    d = xr.open_dataset(pth_way,decode_times=False )
    # Area
    d2 = xr.open_dataset(pth_area,decode_times=False )

    
    ind = np.where((d.time.values >=2170) & (d.time.values <= 2200))
    
    
    tmin = np.min([d.time.size,d2.time.size])
    #melting per area
    y =d2[variable2].values[:tmin] #GT/m^3 yr

   
    yais =y[ind].mean()
  
    
    df['AIS'].iloc[i]= (yais-  y[0])/y[0]*100
   

    for j in range(1,19):
        y =d2[f'{variable2}_sector_'+str(j)].values[:tmin]
        mnan = np.isnan(y)
        
        
       
        df['sector_'+str(j)].iloc[i] = (y[ind].mean()-  y[0])/  y[0]*100
        
    for j in range(1,4):
        y =d2[f'{variable2}_region_'+str(j)].values[:tmin]
        mnan = np.isnan(y)
        
        
       
        df['region_'+str(j)].iloc[i] = (y[ind].mean()-  y[0])/  y[0]*100
    for key in dic_area_of_interest.keys():
        sectors = dic_area_of_interest[key].copy()
        ys = np.zeros((len(sectors),tmin))
        for j,sector in enumerate(sectors):
            ys[j,:] = d2[f'{variable2}{sector}'].values[:tmin].copy()
        sumy = np.sum(ys.copy(),axis = 0)
        df[key].iloc[i]= (np.nanmean(sumy[ind]).copy()-  sumy[0])/  sumy[0]*100
        
        

        
        
#         if (j==1 and i ==4):
#             print(d2['iareafl_sector_'+str(j)].values[:tmin])
#             break
df.to_csv(f'tables/pchange_{variable2}_2200.csv', index=False)



/var/folders/ph/m61jd3r90t190p6qvbps3yqct2hr65/T/ipykernel_67707/2373126780.py:53: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['AIS'].iloc[i]= (yais-  y[0])/y[0]*100
/var/folders/ph/m61jd3r90t190p6qvbps3yqct2hr65/T/ipykernel_67707/2373126780.py:62: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['sector_'+str(j)].iloc[i] = (y[ind].mean()-  y[0])/  y[0]*100
/var/folders/ph/m61jd3r90t190p6qvbps3yqct2hr65/T/ipykernel_67707/2373126780.py:70: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/use

In [16]:
df

,Experiment,Model,Path,sector_1,sector_2,sector_3,sector_4,sector_5,sector_6,sector_7,...,sector_16,sector_17,sector_18,region_1,region_2,region_3,FilchnerRonne,Ross,Amundsen,AIS
0,expAE02,DC_ISSM,/Volumes/CrucialX8/computing/data/antarctica/i...,-7.588189,-7.225841,-0.106049,-3.037040,-16.430058,-0.658328,-0.816935,...,-3.223592,-13.934299,-2.827350,-8.721710,-1.834070,-5.061405,-5.600236,-2.893487,-3.037041,-3.139194
1,expAE02,IGE_ElmerIce,/Volumes/CrucialX8/computing/data/antarctica/i...,-1.014437,-5.157163,-5.940731,-2.736648,-10.752504,-0.213947,-0.548331,...,-3.345532,-6.520560,-0.276236,-6.259692,-1.055110,-2.151440,-3.478687,-1.886407,-2.736645,-1.997522
2,expAE02,ILTS_SICOPOLIS,/Volumes/CrucialX8/computing/data/antarctica/i...,-5.802147,-8.393078,-10.320743,-3.405111,-23.509730,-0.461495,-0.346324,...,-1.071734,-12.371704,-4.082906,-12.132020,-1.821678,-5.687474,-7.582441,-3.060063,-3.405105,-3.696416
3,expAE02,LSCE_GRISLI2,/Volumes/CrucialX8/computing/data/antarctica/i...,-5.803064,-13.064085,-7.685078,-1.270735,-13.547179,-0.669890,0.036175,...,-2.152792,-9.312499,-0.184147,-10.176800,-1.065203,-2.956371,-4.038035,-4.943348,-1.270724,-2.742210
4,expAE02,LSCE_GRISLI,/Volumes/CrucialX8/computing/data/antarctica/i...,-16.528089,-23.530254,-11.170822,-4.311382,-23.646134,-1.456334,-0.720973,...,-6.742132,-45.455745,-11.711583,-18.659768,-2.594833,-17.857502,-7.380207,-9.024914,-4.311381,-5.836747
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
139,expAE05,VUW_PISM2,/Volumes/CrucialX8/computing/data/antarctica/i...,-72.580868,-20.685965,-32.658193,-7.168150,-28.329939,-2.266674,-1.598500,...,-6.795938,-23.066421,-20.973460,-23.489359,-3.557852,-18.426535,-9.827233,-8.630710,-7.168155,-7.551149
140,expAE05,VUW_PISM2_s1,/Volumes/CrucialX8/computing/data/antarctica/i...,-79.455745,-21.308278,-34.209692,-6.212996,-29.461163,-2.269148,-2.287300,...,-6.151519,-24.356443,-21.093042,-24.302785,-3.730865,-18.655021,-10.158114,-8.844375,-6.212989,-7.840621
141,expAE05,VUW_PISM2_s2,/Volumes/CrucialX8/computing/data/antarctica/i...,-70.110905,-20.208631,-30.475163,-9.193091,-27.340803,-2.289460,-1.467631,...,-7.323972,-21.981882,-20.357643,-23.135810,-3.459084,-17.940159,-9.458518,-8.480672,-9.193100,-7.396258
142,expAE05,VUW_PISM2_s3,/Volumes/CrucialX8/computing/data/antarctica/i...,-73.845851,-19.437242,-27.782854,-5.291763,-26.834795,-2.248357,-1.590478,...,-7.362003,-23.823592,-21.158424,-21.899202,-3.358433,-18.817757,-9.219866,-8.186745,-5.291774,-7.114938


In [17]:
# avg and abs 2100
models=[]
paths =[]
exps =[]
for exp in expnames:
    lst = glob.glob(pth_helene+f'{exp}/shelfmelt/*.nc')
    lst.sort()

    for name in lst:
        model = get_model(name,exp)
        if model in removeFileID:
            continue
        else:
            models.append(model)
            paths.append(name)
            exps.append(exp)
df = pd.DataFrame({ 'Experiment': exps,'Model': models, 'Path': paths}) 
for i in range(1,19):
    df['sector_'+str(i)]=0
for i in range(1,4):
    df['region_'+str(i)]=0
for key in dic_area_of_interest.keys():
    df[key]=0
df['AIS']=0
df2 = df.copy()
for i in range (df.shape[0]):
    pth_way = df.iloc[i].Path
    #get shelf area
    pth_area = f'{pth_helene}/{df.iloc[i].Experiment}/iareagr/computed_iareagr_AIS_{df.iloc[i].Model}_{df.iloc[i].Experiment}.nc'
    pth_tf = f'{pth_imip6hack}/{df.iloc[i].Experiment}/thermalforcing/computed_thermalforcing_AIS_{df.iloc[i].Model}_{df.iloc[i].Experiment}.nc'


    #thermal foring 
#     d3 = xr.open_dataset(pth_tf,decode_times=False )

    # Melting
    d = xr.open_dataset(pth_way,decode_times=False )
    # Area
    d2 = xr.open_dataset(pth_area,decode_times=False )

    
    ind = np.where((d.time.values >=2270) & (d.time.values <= 2300))
    
    
    tmin = np.min([d.time.size,d2.time.size])
    #melting per area
    y =d2[variable2].values[:tmin] #GT/m^3 yr

   
    yais =y[ind].mean()
  
    
    df['AIS'].iloc[i]= (yais-  y[0])/y[0]*100
   

    for j in range(1,19):
        y =d2[f'{variable2}_sector_'+str(j)].values[:tmin]
        mnan = np.isnan(y)
        
        
       
        df['sector_'+str(j)].iloc[i] = (y[ind].mean()-  y[0])/  y[0]*100
        
    for j in range(1,4):
        y =d2[f'{variable2}_region_'+str(j)].values[:tmin]
        mnan = np.isnan(y)
        
        
       
        df['region_'+str(j)].iloc[i] = (y[ind].mean()-  y[0])/  y[0]*100
    for key in dic_area_of_interest.keys():
        sectors = dic_area_of_interest[key].copy()
        ys = np.zeros((len(sectors),tmin))
        for j,sector in enumerate(sectors):
            ys[j,:] = d2[f'{variable2}{sector}'].values[:tmin].copy()
        sumy = np.sum(ys.copy(),axis = 0)
        df[key].iloc[i]= (np.nanmean(sumy[ind]).copy()-  sumy[0])/  sumy[0]*100
        
        

        
        
#         if (j==1 and i ==4):
#             print(d2['iareafl_sector_'+str(j)].values[:tmin])
#             break
df.to_csv(f'tables/pchange_{variable2}_2300.csv', index=False)



/var/folders/ph/m61jd3r90t190p6qvbps3yqct2hr65/T/ipykernel_67707/2227998820.py:53: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['AIS'].iloc[i]= (yais-  y[0])/y[0]*100
/var/folders/ph/m61jd3r90t190p6qvbps3yqct2hr65/T/ipykernel_67707/2227998820.py:62: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['sector_'+str(j)].iloc[i] = (y[ind].mean()-  y[0])/  y[0]*100
/var/folders/ph/m61jd3r90t190p6qvbps3yqct2hr65/T/ipykernel_67707/2227998820.py:70: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/use

In [18]:
df

,Experiment,Model,Path,sector_1,sector_2,sector_3,sector_4,sector_5,sector_6,sector_7,...,sector_16,sector_17,sector_18,region_1,region_2,region_3,FilchnerRonne,Ross,Amundsen,AIS
0,expAE02,DC_ISSM,/Volumes/CrucialX8/computing/data/antarctica/i...,-15.889432,-14.195018,-12.781875,-4.881071,-23.504156,-0.941561,-1.375861,...,-9.176544,-16.383046,-5.826819,-15.148379,-2.780950,-8.639662,-7.813758,-5.452209,-4.881074,-5.125958
1,expAE02,IGE_ElmerIce,/Volumes/CrucialX8/computing/data/antarctica/i...,-3.856429,-9.410708,-7.707731,-4.072754,-17.557459,-0.503295,-1.853424,...,-3.810748,-10.644935,-2.628543,-10.478190,-1.611477,-4.436487,-5.512971,-3.516968,-4.072748,-3.239903
2,expAE02,ILTS_SICOPOLIS,/Volumes/CrucialX8/computing/data/antarctica/i...,-16.582407,-27.053428,-12.578948,-12.325508,-31.568542,-0.834049,-1.171380,...,-6.291020,-31.846070,-12.360226,-24.146563,-3.225121,-16.344453,-10.336446,-9.424117,-12.325505,-7.173724
3,expAE02,LSCE_GRISLI2,/Volumes/CrucialX8/computing/data/antarctica/i...,-13.506116,-25.661102,-10.382494,-2.813287,-19.003564,-0.820118,-0.533766,...,-3.922988,-12.448533,-5.266044,-17.553596,-1.677236,-6.850257,-5.495926,-9.385145,-2.813297,-4.651849
4,expAE02,LSCE_GRISLI,/Volumes/CrucialX8/computing/data/antarctica/i...,-47.057262,-40.898934,-24.383701,-18.770050,-30.585840,-1.900393,-1.615757,...,-20.054226,-59.052533,-45.028138,-32.581714,-4.498375,-42.523715,-9.312887,-15.272010,-18.770053,-10.464863
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
139,expAE05,VUW_PISM2,/Volumes/CrucialX8/computing/data/antarctica/i...,-93.069768,-34.723744,-42.501202,-38.274407,-43.015844,-2.276859,-6.373496,...,-23.241368,-43.045002,-40.706030,-41.039571,-5.588515,-37.516659,-14.832039,-13.487571,-38.274398,-12.835945
140,expAE05,VUW_PISM2_s1,/Volumes/CrucialX8/computing/data/antarctica/i...,-93.954867,-37.545845,-43.536153,-42.703578,-44.194674,-2.281909,-7.958649,...,-23.592991,-43.783268,-43.497065,-43.470865,-5.926324,-39.358321,-15.134464,-14.460441,-42.703571,-13.590474
141,expAE05,VUW_PISM2_s2,/Volumes/CrucialX8/computing/data/antarctica/i...,-92.184669,-32.407314,-39.395526,-40.053046,-42.150837,-2.305415,-5.401310,...,-23.583153,-42.763036,-37.390706,-39.969721,-5.381301,-35.624021,-14.270166,-12.705853,-40.053049,-12.428826
142,expAE05,VUW_PISM2_s3,/Volumes/CrucialX8/computing/data/antarctica/i...,-89.583868,-29.599509,-37.444809,-10.134013,-40.146267,-2.271616,-4.790341,...,-23.683342,-44.292057,-37.821659,-32.191059,-5.108656,-36.237517,-13.520892,-11.712794,-10.134013,-10.821393


In [76]:
# avg and abs 2100
models=[]
paths =[]
exps =[]
for exp in expnames:
    lst = glob.glob(pth_helene+f'{exp}/shelfmelt/*.nc')
    lst.sort()

    for name in lst:
        model = get_model(name,exp)
        if model in removeFileID:
            continue
        else:
            models.append(model)
            paths.append(name)
            exps.append(exp)
df = pd.DataFrame({ 'Experiment': exps,'Model': models, 'Path': paths}) 
for i in range(1,19):
    df['sector_'+str(i)]=0
for i in range(1,4):
    df['region_'+str(i)]=0
for key in dic_area_of_interest.keys():
    df[key]=0
df['AIS']=0
df2 = df.copy()
for i in range (df.shape[0]):
    pth_way = df.iloc[i].Path
    #get shelf area
    pth_area = f'{pth_helene}/{df.iloc[i].Experiment}/iareafl/computed_iareafl_AIS_{df.iloc[i].Model}_{df.iloc[i].Experiment}.nc'
    pth_tf = f'{pth_imip6hack}/{df.iloc[i].Experiment}/thermalforcing/computed_thermalforcing_AIS_{df.iloc[i].Model}_{df.iloc[i].Experiment}.nc'


    #thermal foring 
#     d3 = xr.open_dataset(pth_tf,decode_times=False )

    # Melting
    d = xr.open_dataset(pth_way,decode_times=False )
    # Area
    d2 = xr.open_dataset(pth_area,decode_times=False )

    
    ind = np.where((d.time.values >=2170) & (d.time.values <= 2200))
    
    
    tmin = np.min([d.time.size,d2.time.size])
    #melting per area
    y =d2[variable2].values[:tmin] #GT/m^3 yr

   
    yais =y[ind].mean()
  
    
    df['AIS'].iloc[i]=  yais
   

    for j in range(1,19):
        y =d2[f'{variable2}_sector_'+str(j)].values[:tmin]
        mnan = np.isnan(y)
        
        
       
        df['sector_'+str(j)].iloc[i] = y[ind].mean()
        
    for j in range(1,4):
        y =d2[f'{variable2}_region_'+str(j)].values[:tmin]
        mnan = np.isnan(y)
        
        
       
        df['region_'+str(j)].iloc[i] = y[ind].mean()
    for key in dic_area_of_interest.keys():
        sectors = dic_area_of_interest[key].copy()
        ys = np.zeros((len(sectors),tmin))
        for j,sector in enumerate(sectors):
            ys[j,:] = d2[f'{variable2}{sector}'].values[:tmin].copy()
        sumy = np.sum(ys.copy(),axis = 0)
        df[key].iloc[i]= np.nanmean(sumy[ind]).copy()
        
        

        
        
#         if (j==1 and i ==4):
#             print(d2['iareafl_sector_'+str(j)].values[:tmin])
#             break
df.to_csv(f'tables/{variable2}_2200.csv', index=False)


/var/folders/ph/m61jd3r90t190p6qvbps3yqct2hr65/T/ipykernel_33709/777185114.py:53: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['AIS'].iloc[i]=  yais
/var/folders/ph/m61jd3r90t190p6qvbps3yqct2hr65/T/ipykernel_33709/777185114.py:62: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['sector_'+str(j)].iloc[i] = y[ind].mean()
/var/folders/ph/m61jd3r90t190p6qvbps3yqct2hr65/T/ipykernel_33709/777185114.py:70: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-v

In [63]:
df

,Experiment,Model,Path,sector_1,sector_2,sector_3,sector_4,sector_5,sector_6,sector_7,...,sector_15,sector_16,sector_17,sector_18,region_1,region_2,FilchnerRonne,Ross,Amundsen,AIS
0,expAE02,DC_ISSM,/Volumes/CrucialX8/computing/data/antarctica/i...,26208309248,328156479488,26259832832,22817908736,365059866624,234651516928,15723011072,...,90798432256,31774119936,2.349983e+10,22221424640,768502530048,773371592704,5.750019e+11,5.628080e+11,2.281791e+10,1619369656320
1,expAE02,IGE_ElmerIce,/Volumes/CrucialX8/computing/data/antarctica/i...,7965477888,311750721536,19754539008,17412534272,308516552704,234327212032,8893003776,...,36235608064,1925213440,1.580376e+09,10271821824,665399721984,619117084672,4.897472e+11,5.460780e+11,1.741253e+10,1298294243328
2,expAE02,ILTS_SICOPOLIS,/Volumes/CrucialX8/computing/data/antarctica/i...,9000211456,383880855552,13787720704,20843489280,430425702400,237313818624,21111326720,...,81527300096,24202266624,1.805322e+10,6286142464,857938067456,753894490112,6.599313e+11,6.211947e+11,2.084349e+10,1660374220800
3,expAE02,LSCE_GRISLI2,/Volumes/CrucialX8/computing/data/antarctica/i...,9144881152,183743365120,11069212672,4944662016,79312822272,34249181184,6777843200,...,15442131968,6140692480,2.368517e+09,6956079104,288214941696,159454167040,1.168881e+11,2.179925e+11,4.944662e+09,463134359552
4,expAE02,LSCE_GRISLI,/Volumes/CrucialX8/computing/data/antarctica/i...,5140168192,219033714688,5318920704,3866828032,74277167104,23950766080,4235002880,...,12411904000,3666711040,1.032474e+09,4266505216,307636731904,114202476544,9.900766e+10,2.429844e+11,3.866828e+09,430804992000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
139,expAE05,VUW_PISM2,/Volumes/CrucialX8/computing/data/antarctica/i...,6104545280,84449501184,16870647808,16518554624,103329497088,13350931456,6893416448,...,20499472384,5226190848,8.530474e+09,11797696512,227272736768,242738823168,1.420367e+11,9.780043e+10,1.651856e+10,495565864960
140,expAE05,VUW_PISM2_s1,/Volumes/CrucialX8/computing/data/antarctica/i...,7449758208,90810793984,19615975424,16844766208,97609064448,13220431872,9621811200,...,21675065344,4672238592,9.585027e+09,12838450176,232330379264,246820159488,1.363501e+11,1.040312e+11,1.684477e+10,506246266880
141,expAE05,VUW_PISM2_s2,/Volumes/CrucialX8/computing/data/antarctica/i...,5915857920,79341142016,16238454784,17434615808,102961889280,13133936640,6162634752,...,19531264000,5285320704,1.005040e+10,11923969024,221891952640,226930999296,1.405930e+11,9.247508e+10,1.743462e+10,476082634752
142,expAE05,VUW_PISM2_s3,/Volumes/CrucialX8/computing/data/antarctica/i...,7234918912,79586549760,15014815744,12166726656,86025715712,13364265984,7154628608,...,20582791168,4971957760,1.088624e+10,11234804736,200028749824,225796849664,1.206116e+11,9.295081e+10,1.216673e+10,452918706176


In [65]:
22817908736*1.0

22817908736.0

In [66]:
2.281791e+10


22817910000.0

In [77]:
# avg and abs 2100
models=[]
paths =[]
exps =[]
for exp in expnames:
    lst = glob.glob(pth_helene+f'{exp}/shelfmelt/*.nc')
    lst.sort()

    for name in lst:
        model = get_model(name,exp)
        if model in removeFileID:
            continue
        else:
            models.append(model)
            paths.append(name)
            exps.append(exp)
df = pd.DataFrame({ 'Experiment': exps,'Model': models, 'Path': paths}) 
for i in range(1,19):
    df['sector_'+str(i)]=0
for i in range(1,4):
    df['region_'+str(i)]=0
for key in dic_area_of_interest.keys():
    df[key]=0
df['AIS']=0
df2 = df.copy()
for i in range (df.shape[0]):
    pth_way = df.iloc[i].Path
    #get shelf area
    pth_area = f'{pth_helene}/{df.iloc[i].Experiment}/iareafl/computed_iareafl_AIS_{df.iloc[i].Model}_{df.iloc[i].Experiment}.nc'
    pth_tf = f'{pth_imip6hack}/{df.iloc[i].Experiment}/thermalforcing/computed_thermalforcing_AIS_{df.iloc[i].Model}_{df.iloc[i].Experiment}.nc'


    #thermal foring 
#     d3 = xr.open_dataset(pth_tf,decode_times=False )

    # Melting
    d = xr.open_dataset(pth_way,decode_times=False )
    # Area
    d2 = xr.open_dataset(pth_area,decode_times=False )

    
    ind = np.where((d.time.values >=2270) & (d.time.values <= 2300))
    
    
    tmin = np.min([d.time.size,d2.time.size])
    #melting per area
    y =d2[variable2].values[:tmin] #GT/m^3 yr

   
    yais =y[ind].mean()
  
    
    df['AIS'].iloc[i]=  yais
   

    for j in range(1,19):
        y =d2[f'{variable2}_sector_'+str(j)].values[:tmin]
        mnan = np.isnan(y)
        
        
       
        df['sector_'+str(j)].iloc[i] = y[ind].mean()
        
    for j in range(1,4):
        y =d2[f'{variable2}_region_'+str(j)].values[:tmin]
        mnan = np.isnan(y)
        
        
       
        df['region_'+str(j)].iloc[i] = y[ind].mean()
    for key in dic_area_of_interest.keys():
        sectors = dic_area_of_interest[key].copy()
        ys = np.zeros((len(sectors),tmin))
        for j,sector in enumerate(sectors):
            ys[j,:] = d2[f'{variable2}{sector}'].values[:tmin].copy()
        sumy = np.sum(ys.copy(),axis = 0)
        df[key].iloc[i]= np.nanmean(sumy[ind]).copy()
        
        

        
        
#         if (j==1 and i ==4):
#             print(d2['iareafl_sector_'+str(j)].values[:tmin])
#             break
df.to_csv(f'tables/{variable2}_2300.csv', index=False)


/var/folders/ph/m61jd3r90t190p6qvbps3yqct2hr65/T/ipykernel_33709/205797514.py:53: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['AIS'].iloc[i]=  yais
/var/folders/ph/m61jd3r90t190p6qvbps3yqct2hr65/T/ipykernel_33709/205797514.py:62: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['sector_'+str(j)].iloc[i] = y[ind].mean()
/var/folders/ph/m61jd3r90t190p6qvbps3yqct2hr65/T/ipykernel_33709/205797514.py:70: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-v